# Real-Time Driving Perception Inference — Runner Notebook

Run this notebook on **Google Colab with a GPU runtime** (Runtime → Change runtime type → T4 GPU).

Each phase has its own section. Run sections top-to-bottom in a single session.

## 0. Configuration

Edit these variables before running anything else.

In [ ]:
# ── Fill in your values ──────────────────────────────────────────────────────
REPO_URL         = 'https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git'
REPO_DIR         = 'real-time-driving-inference-optimizer'   # folder cloned into /content
DRIVE_VIDEO_PATH = '/content/drive/MyDrive/clip.mp4'         # path to video on your Drive
# ─────────────────────────────────────────────────────────────────────────────

import os
WORKSPACE = f'/content/{REPO_DIR}'
print(f'Workspace : {WORKSPACE}')
print(f'Repo URL  : {REPO_URL}')

## 1. Setup — Mount Drive, Clone Repo, Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

if os.path.exists(WORKSPACE):
    print('Repo already cloned — pulling latest changes...')
    !git -C {WORKSPACE} pull
else:
    !git clone {REPO_URL} {WORKSPACE}

%cd {WORKSPACE}
!pwd

In [ ]:
!pip install -q -r requirements.txt

## 2. Environment Check

In [ ]:
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'CUDA version    : {torch.version.cuda}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Copy Video from Drive

In [ ]:
import shutil, os

dest = 'data/clip.mp4'
os.makedirs('data', exist_ok=True)

if not os.path.exists(dest):
    print(f'Copying {DRIVE_VIDEO_PATH} → {dest} ...')
    shutil.copy(DRIVE_VIDEO_PATH, dest)
    print('Done.')
else:
    print(f'{dest} already present.')

size_mb = os.path.getsize(dest) / 1e6
print(f'File size: {size_mb:.1f} MB')

## Phase 1 — PyTorch Baseline Inference

Runs YOLO11n on every frame. Records per-frame latency for each pipeline stage:
read → preprocess → inference → postprocess → draw+write.

Outputs:
- `results/pytorch_output.mp4` — annotated video
- `results/pytorch_raw_timings.csv` — per-frame timings

In [ ]:
!python3 scripts/run_pytorch_video.py \
    --video data/clip.mp4 \
    --output-video results/pytorch_output.mp4 \
    --results results/pytorch_raw_timings.csv \
    --model yolo11n.pt \
    --input-size 640 \
    --conf 0.25 \
    --iou 0.45 \
    --warmup 10 \
    --device cuda

### Phase 1 Results Preview

In [ ]:
import pandas as pd
import sys
sys.path.insert(0, '.')
from src.metrics import compute_stats, fps_from_mean_ms

df = pd.read_csv('results/pytorch_raw_timings.csv')
print(f'Frames recorded: {len(df)}')
print()

stages = ['read_ms', 'preprocess_ms', 'inference_ms', 'postprocess_ms', 'draw_write_ms', 'total_ms']
summary = []
for col in stages:
    s = compute_stats(df[col].tolist())
    summary.append({
        'stage':   col.replace('_ms', ''),
        'mean_ms': round(s['mean_ms'], 2),
        'p50_ms':  round(s['p50_ms'],  2),
        'p90_ms':  round(s['p90_ms'],  2),
        'p99_ms':  round(s['p99_ms'],  2),
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
print()
total_mean = df['total_ms'].mean()
print(f'End-to-end FPS: {fps_from_mean_ms(total_mean):.1f}')

### Download Results

Download the CSV and annotated video before ending the session.

In [ ]:
from google.colab import files

files.download('results/pytorch_raw_timings.csv')
files.download('results/pytorch_output.mp4')

---
## Phase 2 — Benchmarking

Aggregates the Phase 1 raw timings into a summary CSV with mean / p50 / p90 / p99
per pipeline stage, then generates four charts:
- `plots/pipeline_breakdown.png` — stacked bar of mean stage times
- `plots/latency_percentiles.png` — p50 / p90 / p99 per stage
- `plots/fps_comparison.png` — FPS bar chart
- `plots/latency_timeline.png` — per-frame total latency over time

Output: `results/pytorch_baseline.csv`

In [ ]:
!python3 scripts/benchmark.py \
    --results results/pytorch_raw_timings.csv \
    --output  results/pytorch_baseline.csv \
    --runtime pytorch

In [ ]:
!python3 scripts/plot_results.py \
    --baselines results/pytorch_baseline.csv \
    --raw       results/pytorch_raw_timings.csv \
    --labels    pytorch \
    --output-dir plots/

### Phase 2 — Inline Chart Preview

In [ ]:
from IPython.display import Image, display

for chart in ['pipeline_breakdown', 'latency_percentiles', 'fps_comparison', 'latency_timeline']:
    path = f'plots/{chart}.png'
    print(f'── {chart} ──')
    display(Image(filename=path))

### Phase 2 — Download Results

In [ ]:
from google.colab import files

files.download('results/pytorch_baseline.csv')
for chart in ['pipeline_breakdown', 'latency_percentiles', 'fps_comparison', 'latency_timeline']:
    files.download(f'plots/{chart}.png')

---
## Phase 3 — ONNX Export and Validation

Exports YOLO11n to ONNX, then runs ORT inference on the full video with
identical 5-stage timing. Also validates output consistency vs PyTorch
(raw diff, detection count match rate, box/confidence drift).

Outputs:
- `models/yolo11n.onnx` — exported ONNX model
- `results/onnx_output.mp4` — annotated video
- `results/onnx_raw_timings.csv` — per-frame timings
- `results/onnx_benchmark.csv` — summary stats

In [ ]:
!python3 scripts/export_onnx.py \
    --model      yolo11n.pt \
    --output     models/yolo11n.onnx \
    --input-size 640 \
    --validate \
    --video      data/clip.mp4 \
    --val-frames 20 \
    --device     cuda

In [ ]:
!python3 scripts/run_onnx_video.py \
    --video        data/clip.mp4 \
    --onnx-model   models/yolo11n.onnx \
    --pt-model     yolo11n.pt \
    --output-video results/onnx_output.mp4 \
    --results      results/onnx_raw_timings.csv \
    --input-size   640 \
    --conf         0.25 \
    --iou          0.45 \
    --warmup       10 \
    --device       cuda

In [ ]:
!python3 scripts/benchmark.py \
    --results results/onnx_raw_timings.csv \
    --output  results/onnx_benchmark.csv \
    --runtime onnx

### Phase 3 — PyTorch vs ONNX Comparison Charts

In [ ]:
!python3 scripts/plot_results.py \
    --baselines results/pytorch_baseline.csv results/onnx_benchmark.csv \
    --labels    pytorch onnx \
    --output-dir plots/

In [ ]:
from IPython.display import Image, display

for chart in ['pipeline_breakdown', 'latency_percentiles', 'fps_comparison']:
    print(f'── {chart} ──')
    display(Image(filename=f'plots/{chart}.png'))

### Phase 3 — Download Results

In [ ]:
from google.colab import files

files.download('results/onnx_raw_timings.csv')
files.download('results/onnx_benchmark.csv')
files.download('results/onnx_output.mp4')
for chart in ['pipeline_breakdown', 'latency_percentiles', 'fps_comparison']:
    files.download(f'plots/{chart}.png')

---
## Phase 4 — TensorRT Optimization

Builds FP32 and FP16 TRT engines from the ONNX model, then runs both on the
full video with the same 5-stage timing. Engine build is hardware-specific and
**must be rebuilt at the start of every new Colab session**.

Build times on T4:  FP32 ≈ 2–5 min  /  FP16 ≈ 3–7 min

Outputs:
- `models/yolo11n_fp32.engine` / `models/yolo11n_fp16.engine`
- `results/trt_fp32_raw_timings.csv` / `results/trt_fp16_raw_timings.csv`
- `results/trt_fp32_benchmark.csv`  / `results/trt_fp16_benchmark.csv`

---
## Phase 5 — Pipeline Bottleneck Optimization

Two targeted sweeps based on Phase 2 profiling findings:

**Optimization 1 — Resolution sweep (PyTorch)**
Tests 320×320, 480×480, 640×640 inputs. Quantifies the speed vs detection
quality tradeoff — no model rebuild needed, letterbox handles any size.

**Optimization 2 — Confidence threshold sweep (TRT FP16)**
Tests conf ∈ {0.05, 0.15, 0.25, 0.50, 0.75} on the best runtime. Higher
thresholds pre-filter more boxes before NMS, reducing postprocessing time
at the cost of potentially fewer detections.

Outputs:
- `results/resolution_sweep.csv`   +  `plots/resolution_sweep.png`
- `results/conf_sweep.csv`         +  `plots/conf_sweep.png`

### Phase 5 — Optimization 1: Resolution Sweep (PyTorch)

In [ ]:
!python3 scripts/optimize_resolution.py \
    --video       data/clip.mp4 \
    --model       yolo11n.pt \
    --resolutions 320 480 640 \
    --conf        0.25 \
    --iou         0.45 \
    --warmup      10 \
    --results     results/resolution_sweep.csv \
    --plot        plots/resolution_sweep.png \
    --device      cuda

In [ ]:
from IPython.display import Image, display
display(Image(filename='plots/resolution_sweep.png'))

### Phase 5 — Optimization 2: Confidence Threshold Sweep (TRT FP16)

In [ ]:
!python3 scripts/optimize_conf_threshold.py \
    --video       data/clip.mp4 \
    --engine      models/yolo11n_fp16.engine \
    --conf-values 0.05 0.15 0.25 0.50 0.75 \
    --iou         0.45 \
    --warmup      10 \
    --input-size  640 \
    --results     results/conf_sweep.csv \
    --plot        plots/conf_sweep.png

In [ ]:
from IPython.display import Image, display
display(Image(filename='plots/conf_sweep.png'))

### Phase 5 — Download Results

In [ ]:
from google.colab import files

files.download('results/resolution_sweep.csv')
files.download('results/conf_sweep.csv')
files.download('plots/resolution_sweep.png')
files.download('plots/conf_sweep.png')

### Phase 4 — Build Engines (FP32 + FP16)

In [ ]:
!python3 scripts/build_tensorrt_engine.py \
    --onnx         models/yolo11n.onnx \
    --output-dir   models/ \
    --fp32 --fp16 \
    --workspace-gb 2

### Phase 4 — Run FP32 Inference

In [ ]:
!python3 scripts/run_tensorrt_video.py \
    --video        data/clip.mp4 \
    --engine       models/yolo11n_fp32.engine \
    --pt-model     yolo11n.pt \
    --output-video results/trt_fp32_output.mp4 \
    --results      results/trt_fp32_raw_timings.csv \
    --input-size   640 \
    --conf         0.25 \
    --iou          0.45 \
    --warmup       10

In [ ]:
!python3 scripts/benchmark.py \
    --results results/trt_fp32_raw_timings.csv \
    --output  results/trt_fp32_benchmark.csv \
    --runtime trt-fp32

### Phase 4 — Run FP16 Inference

In [ ]:
!python3 scripts/run_tensorrt_video.py \
    --video        data/clip.mp4 \
    --engine       models/yolo11n_fp16.engine \
    --pt-model     yolo11n.pt \
    --output-video results/trt_fp16_output.mp4 \
    --results      results/trt_fp16_raw_timings.csv \
    --input-size   640 \
    --conf         0.25 \
    --iou          0.45 \
    --warmup       10

In [ ]:
!python3 scripts/benchmark.py \
    --results results/trt_fp16_raw_timings.csv \
    --output  results/trt_fp16_benchmark.csv \
    --runtime trt-fp16

### Phase 4 — Full 4-Runtime Comparison Charts

Regenerates all comparison charts with all four runtimes side by side.

In [ ]:
!python3 scripts/plot_results.py \
    --baselines results/pytorch_baseline.csv \
                results/onnx_benchmark.csv \
                results/trt_fp32_benchmark.csv \
                results/trt_fp16_benchmark.csv \
    --labels    pytorch onnx trt-fp32 trt-fp16 \
    --output-dir plots/

In [ ]:
from IPython.display import Image, display

for chart in ['pipeline_breakdown', 'latency_percentiles', 'fps_comparison']:
    print(f'── {chart} ──')
    display(Image(filename=f'plots/{chart}.png'))

### Phase 4 — Download Results

In [ ]:
from google.colab import files

for f in ['results/trt_fp32_raw_timings.csv', 'results/trt_fp32_benchmark.csv',
          'results/trt_fp16_raw_timings.csv', 'results/trt_fp16_benchmark.csv']:
    files.download(f)

for chart in ['pipeline_breakdown', 'latency_percentiles', 'fps_comparison']:
    files.download(f'plots/{chart}.png')

## Phase 5 — Bottleneck Optimization
> Coming in Phase 5 development.